In [1]:
import dotenv

import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterstats

from rasterstats import zonal_stats
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.mask import mask

from food_security import salinity_correction, water_quality

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / "OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/"

In [3]:
def count_per_type(x, field_type):
    x = x[x == field_type]
    return np.ma.count(x)


def read_field_sizes(tif_file, area_gdf):
    with rasterio.open(tif_file) as src:
        stats = rasterstats.zonal_stats(
            area_gdf,
            tif_file,
            stats="count",
            add_stats={
                "count_0": lambda x: count_per_type(x, 0),
                "count_1": lambda x: count_per_type(x, 3502),
                "count_2": lambda x: count_per_type(x, 3503),
                "count_3": lambda x: count_per_type(x, 3504),
                "count_4": lambda x: count_per_type(x, 3505),
                "count_5": lambda x: count_per_type(x, 3506),
            },
            nodata=src.nodata,
        )

    return stats


def create_stats_df(stats, area_gdf, area_column="Name"):
    df_dict = {
        "area": area_gdf[area_column].values,
        "nr_of_fields": [s["count"] for s in stats],
        "nr_no_field": [s["count_0"] for s in stats],
        "nr_<0.64ha": [s["count_5"] for s in stats],
        "nr_0.64-2.56ha": [s["count_4"] for s in stats],
        "nr_2.56-16ha": [s["count_3"] for s in stats],
        "nr_16-100ha": [s["count_2"] for s in stats],
        "nr_>100ha": [s["count_1"] for s in stats],
    }
    df = pd.DataFrame(df_dict)
    return df


def compute_fraction(df, field):
    total = df["nr_of_fields"]
    value = df[f"nr_{field}"]

    df[f"fraction_{field}"] = value / total
    return df

In [4]:
field_type_file = src_dir / "dominant_field_size_categories.tif"

com_gdf = gpd.read_file(src_dir / 'Final2_Command_Area.shp')
com_gdf = com_gdf.to_crs("EPSG:4326")

field_stats = read_field_sizes(field_type_file, com_gdf)

field_size_df = create_stats_df(stats=field_stats, area_gdf=com_gdf)
field_size_df['fraction_small'] = (field_size_df['nr_<0.64ha']) / field_size_df['nr_of_fields']
field_size_df['fraction_medium'] = (field_size_df['nr_0.64-2.56ha']) / field_size_df['nr_of_fields']
field_size_df['fraction_large'] = (field_size_df['nr_2.56-16ha']) / field_size_df['nr_of_fields']

In [5]:
with rasterio.open(field_type_file) as src:

    transform, width, height = calculate_default_transform(
        src.crs,
        "EPSG:32636",
        src.width,
        src.height,
        *src.bounds,
    )

    data = np.empty((height, width), dtype=src.dtypes[0])

    reproject(
        source=rasterio.band(src, 1),
        destination=data,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=transform,
        dst_crs="EPSG:32636",
        resampling=Resampling.nearest,
    )

pixel_area_km2 = transform.a * abs(transform.e) / 1e6

field_size_df['arable_km2'] = (field_size_df['nr_of_fields'] - field_size_df['nr_no_field']) * pixel_area_km2

In [6]:
excel_path = "/Users/hemert/OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt_ERF_data/data_correlation.xlsx"

def write_excel_file(df, excel_path, sheet_name, append=False):
    # Open Excel file
    try:
        # book = load_workbook(excel_path)
        with pd.ExcelWriter(
            excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace"
        ) as writer:
            # excel_file.book = book
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    except Exception as e:
        print(e)
        df.to_excel(excel_path, sheet_name=sheet_name, index=False)

In [7]:
command_gdf = gpd.read_file(src_dir / 'Final2_Command_Area.shp')
conversion_df = pd.read_excel(excel_path, sheet_name='command_area')

mapping = (
    command_gdf.merge(
        conversion_df[['area_map_name', 'area_name']],
        left_on='OBJECTID',
        right_on='area_map_name'
    )
    .set_index('Name')['area_name']
)

field_size_df['area'] = field_size_df['area'].map(mapping)

In [8]:
write_excel_file(field_size_df, excel_path, sheet_name="field_sizes")